In [1]:
!pip -q install -U sentence-transformers faiss-cpu pyarrow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 34.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 85.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 31.9 MB/s eta 0:00:00:00:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.


In [2]:
import os
import json
import time

import numpy as np
import pandas as pd
import faiss
import torch

from pathlib import Path
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer

OUT_DIR = Path("/kaggle/working/step11")
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("PyTorch:", torch.__version__)
print("FAISS:", faiss.__version__)
print("GPU available:", torch.cuda.is_available())
print("Output:", OUT_DIR)

PyTorch: 2.10.0+cu128
FAISS: 1.15.1
GPU available: True
Output: /kaggle/working/step11


In [3]:
INPUT_DIR = Path("/kaggle/input")

product_files = list(
    INPUT_DIR.rglob("unified_products.parquet")
)

print("Các file tìm thấy:")

for p in product_files:
    print(p)

assert len(product_files) > 0, (
    "Không tìm thấy unified_products.parquet. "
    "Hãy kiểm tra Add Input Step 10."
)

PRODUCTS_PATH = product_files[0]

print("\nSử dụng:", PRODUCTS_PATH)

Các file tìm thấy:
/kaggle/input/datasets/trnquangtriu/11dataset/unified_products.parquet

Sử dụng: /kaggle/input/datasets/trnquangtriu/11dataset/unified_products.parquet


In [4]:
products = pd.read_parquet(PRODUCTS_PATH)

print("SHAPE:", products.shape)

print("\nCOLUMNS:")
print(products.columns.tolist())

assert len(products) == 164_926, (
    "Số lượng sản phẩm khác Step 10. "
    "Cần kiểm tra lại file đầu vào."
)

assert "product_id" in products.columns

assert products["product_id"].notna().all()

assert products["product_id"].is_unique

print("\nNguồn sản phẩm:")

display(
    products["source"].value_counts(
        dropna=False
    )
)

print("\n5 sản phẩm mẫu:")

sample_cols = [
    col for col in [
        "product_id",
        "product_name",
        "description",
        "product_features",
        "category_path",
        "price",
        "currency",
        "rating"
    ]
    if col in products.columns
]

display(
    products[sample_cols].head()
)

SHAPE: (164926, 60)

COLUMNS:
['product_id', 'product_name', 'category', 'subcategory', 'brand', 'price', 'currency', 'average_rating', 'review_count', 'sold_count', 'description', 'image_url', 'product_url', 'source', 'language', 'smart_product_name', 'smart_source', 'smart_currency', 'smart_price', 'smart_rating', 'smart_review_count', 'product_family', 'product_type', 'classification_method', 'matched_rule', 'has_valid_price', 'has_valid_currency', 'price_filter_eligible', 'quality_issue_count', 'quality_flags_json', 'product_family_v1', 'product_type_v1', 'classification_method_v1', 'matched_rule_v1', 'product_family_v2', 'product_type_v2', 'classification_method_v2', 'classification_reason_v2', 'etl_review_rows', 'etl_review_text_rows', 'etl_review_evidence_rows', 'has_review_evidence', 'product_url_available', 'product_type_candidate', 'product_family_candidate', 'classification_requires_validation', 'top5_evidence_ready', 'asin', 'meta_title', 'meta_description', 'meta_features'

source
amazon    121917
tiki       41575
shopee      1434
Name: count, dtype: int64


5 sản phẩm mẫu:


,product_id,product_name,description,product_features,category_path,price,currency
0,amazon_B07PXGQC1Q,Apple AirPods (2nd Generation) Wireless Earbud...,None,,,99.00,USD
1,amazon_B07KTYJ769,"Amazon Smart Plug, for home automation, Works ...",None,Amazon Smart Plug works with Alexa to add voic...,,24.99,USD
2,amazon_B0BGNG1294,"Amazon Basics HDMI Cable, 18Gbps High-Speed, 4...",Product Description\nAmazon Basics High-Speed ...,IN THE BOX: HDMI cable (A Male to A Male) for ...,Electronics\nTelevision & Video\nAccessories\n...,8.55,USD
3,amazon_B08CLNX58K,SanDisk 32GB 2-Pack Ultra MicroSDHC UHS-I Memo...,Transfer speeds of up to 98MB/sec . Records Fu...,Speeds up to 98MB/sec | Up to 98MB/s read spee...,Electronics\nComputers & Accessories\nComputer...,13.40,USD
4,amazon_B08WJSHSLC,Fire TV Stick (3rd Gen) with Alexa Voice Remot...,None,This bundle includes a Fire TV Stick with Alex...,,44.98,USD


In [5]:
import re
import pandas as pd

TEXT_COLUMNS = [
    "product_name",
    "brand",
    "category",
    "subcategory",
    "category_path",
    "product_features",
    "description",
]

assert all(col in products.columns for col in TEXT_COLUMNS)

INVALID_TEXT = {
    "",
    "none",
    "null",
    "nan",
    "n/a",
    "unknown",
}

def clean_text(value):
    if value is None:
        return ""

    try:
        if pd.isna(value):
            return ""
    except (ValueError, TypeError):
        pass

    text = str(value).strip()

    if text.lower() in INVALID_TEXT:
        return ""

    text = re.sub(r"\s+", " ", text)

    return text


def build_retrieval_text(row):
    sections = []

    fields = [
        ("Product", "product_name", 300),
        ("Brand", "brand", 100),
        ("Category", "category", 150),
        ("Subcategory", "subcategory", 150),
        ("Category path", "category_path", 250),
        ("Features", "product_features", 700),
        ("Description", "description", 700),
    ]

    for label, column, max_chars in fields:
        value = clean_text(row[column])

        if value:
            sections.append(
                f"{label}: {value[:max_chars]}"
            )

    return "\n".join(sections)


products["retrieval_text"] = products.apply(
    build_retrieval_text,
    axis=1
)

print("TOTAL PRODUCTS:", len(products))

print(
    "EMPTY RETRIEVAL TEXT:",
    products["retrieval_text"]
    .str.strip()
    .eq("")
    .sum()
)

print("\nSAMPLE RETRIEVAL TEXT:\n")

print(
    products.loc[3, "retrieval_text"]
)

TOTAL PRODUCTS: 164926
EMPTY RETRIEVAL TEXT: 0

SAMPLE RETRIEVAL TEXT:

Product: SanDisk 32GB 2-Pack Ultra MicroSDHC UHS-I Memory Card (2x32GB) - SDSQUAR-032G-GN6MT
Brand: SanDisk
Category: Computers
Category path: Electronics Computers & Accessories Computer Accessories & Peripherals Memory Cards Micro SD Cards
Features: Speeds up to 98MB/sec | Up to 98MB/s read speed; write speed lower. Based on internal testing; performance may be lower depending on host device, interface, usage, conditions and other factors. 1MB = 1,000,000 bytes Application Performance Class 1 (A1)(1) | (1)Results may vary based on host device, app type and other factors. C10 for Full HD Video(2), UHS Speed Class 1 (U1)(3) | (2)Compatible device required. Full HD (1920x1080) video support may vary based upon host device, file attributes, and other factors.| (3)UHS Speed Class 1 (U1) designates a performance option designed to support real-time video recording with UHS enabled host devices. SanDisk Memory Zone incl

# Khởi tạo model embedding

In [6]:
from sentence_transformers import SentenceTransformer
import torch

MODEL_NAME = (
    "sentence-transformers/"
    "paraphrase-multilingual-MiniLM-L12-v2"
)

DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

model = SentenceTransformer(
    MODEL_NAME,
    device=DEVICE
)

print("MODEL:", MODEL_NAME)
print("DEVICE:", DEVICE)

print(
    "EMBEDDING DIMENSION:",
    model.get_sentence_embedding_dimension()
)

print(
    "MAX SEQUENCE LENGTH:",
    model.max_seq_length
)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

MODEL: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
DEVICE: cuda
EMBEDDING DIMENSION: 384
MAX SEQUENCE LENGTH: 128


/tmp/ipykernel_58/1833651731.py:25: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  model.get_sentence_embedding_dimension()


# Test embedding 5 sản phẩm

In [7]:
import numpy as np

sample_texts = (
    products["retrieval_text"]
    .head(5)
    .tolist()
)

sample_embeddings = model.encode(
    sample_texts,
    batch_size=5,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

print(
    "SAMPLE EMBEDDING SHAPE:",
    sample_embeddings.shape
)

print(
    "DTYPE:",
    sample_embeddings.dtype
)

print(
    "VECTOR NORMS:",
    np.linalg.norm(
        sample_embeddings,
        axis=1
    )
)

assert sample_embeddings.shape == (5, 384)

assert np.isfinite(
    sample_embeddings
).all()

print("PASS: Embedding hoạt động bình thường.")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

SAMPLE EMBEDDING SHAPE: (5, 384)
DTYPE: float32
VECTOR NORMS: [1.         0.99999994 1.         1.         0.99999994]
PASS: Embedding hoạt động bình thường.


# Tạo văn bản embedding ngắn, ưu tiên thông tin quan trọng

In [8]:
def build_embedding_text(row):
    name = clean_text(row["product_name"])
    category = clean_text(row["category_path"])
    if not category:
        category = clean_text(row["category"])

    features = clean_text(row["product_features"])
    description = clean_text(row["description"])

    parts = []

    if name:
        parts.append(name[:220])

    if category:
        parts.append(f"Category: {category[:130]}")

    if features:
        parts.append(f"Features: {features[:240]}")

    if description:
        parts.append(f"Description: {description[:180]}")

    return " | ".join(parts)


products["embedding_text"] = products.apply(
    build_embedding_text,
    axis=1
)

print("TOTAL:", len(products))
print(
    "EMPTY:",
    products["embedding_text"].str.strip().eq("").sum()
)

print("\nSAMPLE:\n")
print(products.loc[3, "embedding_text"])

TOTAL: 164926
EMPTY: 0

SAMPLE:

SanDisk 32GB 2-Pack Ultra MicroSDHC UHS-I Memory Card (2x32GB) - SDSQUAR-032G-GN6MT | Category: Electronics Computers & Accessories Computer Accessories & Peripherals Memory Cards Micro SD Cards | Features: Speeds up to 98MB/sec | Up to 98MB/s read speed; write speed lower. Based on internal testing; performance may be lower depending on host device, interface, usage, conditions and other factors. 1MB = 1,000,000 bytes Application Performance  | Description: Transfer speeds of up to 98MB/sec . Records Full HD video(2). SanDisk Memory Zone app included for easy file management(4). Shock, temperature, water, and X-ray proof (6). 1GB = 1,


# Kiểm tra mức độ bị cắt token

In [9]:
import numpy as np

sample_texts = products["embedding_text"].sample(
    n=min(2000, len(products)),
    random_state=42
).tolist()

token_lengths = [
    len(model.tokenizer.encode(
        text,
        add_special_tokens=True,
        truncation=False
    ))
    for text in sample_texts
]

token_lengths = np.array(token_lengths)

print("MAX MODEL TOKENS:", model.max_seq_length)
print("MEAN TOKENS:", round(token_lengths.mean(), 2))
print("MEDIAN TOKENS:", np.median(token_lengths))
print("P95 TOKENS:", np.percentile(token_lengths, 95))
print("MAX TOKENS:", token_lengths.max())

truncated = (token_lengths > model.max_seq_length).sum()

print(
    "SAMPLE TRUNCATED:",
    truncated,
    "/",
    len(token_lengths)
)

print(
    "TRUNCATION RATE:",
    round(
        truncated / len(token_lengths) * 100,
        2
    ),
    "%"
)

Token indices sequence length is longer than the specified maximum sequence length for this model (150 > 128). Running this sequence through the model will result in indexing errors


MAX MODEL TOKENS: 128
MEAN TOKENS: 137.06
MEDIAN TOKENS: 139.0
P95 TOKENS: 209.04999999999995
MAX TOKENS: 318
SAMPLE TRUNCATED: 1170 / 2000
TRUNCATION RATE: 58.5 %


# Tạo embedding text theo ngân sách token

In [10]:
TOKENIZER = model.tokenizer
MAX_TOKENS = model.max_seq_length

# Mỗi ngân sách bao gồm cả nhãn Category/Features/Description
TOKEN_BUDGETS = {
    "name": 40,
    "category": 18,
    "features": 27,
    "description": 30,
}

def truncate_to_tokens(text, max_tokens):
    if not text or max_tokens <= 0:
        return ""

    token_ids = TOKENIZER.encode(
        text,
        add_special_tokens=False,
        truncation=False
    )

    token_ids = token_ids[:max_tokens]

    return TOKENIZER.decode(
        token_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True
    ).strip()


def build_token_aware_text(row):
    name = clean_text(row["product_name"])

    category = clean_text(row["category_path"])
    if not category:
        category = clean_text(row["category"])

    features = clean_text(row["product_features"])
    description = clean_text(row["description"])

    sections = [
        (
            name,
            TOKEN_BUDGETS["name"]
        ),
        (
            f"Category: {category}" if category else "",
            TOKEN_BUDGETS["category"]
        ),
        (
            f"Features: {features}" if features else "",
            TOKEN_BUDGETS["features"]
        ),
        (
            f"Description: {description}" if description else "",
            TOKEN_BUDGETS["description"]
        ),
    ]

    parts = []

    for text, budget in sections:
        shortened = truncate_to_tokens(
            text,
            budget
        )

        if shortened:
            parts.append(shortened)

    combined = " | ".join(parts)

    # Giới hạn cuối cùng để tính cả token của dấu phân cách.
    # Chừa 2 token đặc biệt của model: [CLS] và [SEP].
    combined = truncate_to_tokens(
        combined,
        MAX_TOKENS - 2
    )

    return combined

# Kiểm tra trên 2.000 sản phẩm trước

In [11]:
sample = products.sample(
    n=2000,
    random_state=42
).copy()

sample["embedding_text_v2"] = sample.apply(
    build_token_aware_text,
    axis=1
)

token_lengths_v2 = [
    len(
        TOKENIZER.encode(
            text,
            add_special_tokens=True,
            truncation=False
        )
    )
    for text in sample["embedding_text_v2"]
]

token_lengths_v2 = np.array(
    token_lengths_v2
)

print("MAX MODEL TOKENS:", MAX_TOKENS)

print(
    "MEAN TOKENS:",
    round(token_lengths_v2.mean(), 2)
)

print(
    "P95 TOKENS:",
    np.percentile(
        token_lengths_v2,
        95
    )
)

print(
    "MAX TOKENS:",
    token_lengths_v2.max()
)

truncated = (
    token_lengths_v2 > MAX_TOKENS
).sum()

print(
    "TRUNCATION RATE:",
    round(
        truncated / len(sample) * 100,
        2
    ),
    "%"
)

assert truncated == 0, (
    "Vẫn có văn bản vượt giới hạn token."
)

assert (
    sample["embedding_text_v2"]
    .str.strip()
    .ne("")
    .all()
)

print("\nSAMPLE:\n")

print(
    sample.iloc[0]["embedding_text_v2"]
)

print("\nPASS: Token-aware embedding text.")

MAX MODEL TOKENS: 128
MEAN TOKENS: 87.78
P95 TOKENS: 123.0
MAX TOKENS: 123
TRUNCATION RATE: 0.0 %

SAMPLE:

mCover iPearl Hard Shell Case for 2017 11.6" Dell Chromebook 11 3189 Series 2-in-1 Laptop (NOT Compatible with 210-ACDU / 3120 | Category: Electronics Computers & Accessories Laptop Accessories Bags, Case | Features: Made of high-quality translucent polycarbonate material, which is shatter-proof and will protect your

PASS: Token-aware embedding text.


# Tạo embedding text cho toàn bộ sản phẩm

In [12]:
import time
import numpy as np
import pandas as pd

start_time = time.time()

products["embedding_text_v2"] = products.apply(
    build_token_aware_text,
    axis=1
)

assert len(products) == 164_926

assert products["product_id"].is_unique

assert (
    products["embedding_text_v2"]
    .str.strip()
    .ne("")
    .all()
)

print("TOTAL PRODUCTS:", len(products))

print(
    "EMPTY EMBEDDING TEXT:",
    products["embedding_text_v2"]
    .str.strip()
    .eq("")
    .sum()
)

print(
    "TIME:",
    round(time.time() - start_time, 2),
    "seconds"
)

print("\nSAMPLE:\n")
print(products.loc[3, "embedding_text_v2"])

TOTAL PRODUCTS: 164926
EMPTY EMBEDDING TEXT: 0
TIME: 280.06 seconds

SAMPLE:

SanDisk 32GB 2-Pack Ultra MicroSDHC UHS-I Memory Card (2x32GB) - SDSQUAR-032G-GN6MT | Category: Electronics Computers & Accessories Computer Accessories & Peripherals | Features: Speeds up to 98MB/sec | Up to 98MB/s read speed; write speed lower. Base | Description: Transfer speeds of up to 98MB/sec. Records Full HD video(2). SanDisk Memory Zone app included for easy file


# Lưu metadata và nội dung embedding

In [13]:
from pathlib import Path

OUT_DIR = Path("/kaggle/working/step11")
OUT_DIR.mkdir(parents=True, exist_ok=True)

metadata_cols = [
    "product_id",
    "product_name",
    "source",
    "category",
    "subcategory",
    "category_path",
    "brand",
    "price",
    "currency",
    "average_rating",
    "review_count",
    "description",
    "product_features",
    "product_url",
    "retrieval_text",
    "embedding_text_v2",
]

metadata_cols = [
    col
    for col in metadata_cols
    if col in products.columns
]

product_metadata = products[
    metadata_cols
].copy().reset_index(drop=True)

product_metadata.insert(
    0,
    "faiss_row_id",
    np.arange(
        len(product_metadata),
        dtype=np.int64
    )
)

METADATA_PATH = (
    OUT_DIR / "product_embedding_metadata.parquet"
)

product_metadata.to_parquet(
    METADATA_PATH,
    index=False
)

print("SAVED:", METADATA_PATH)

print(
    "SHAPE:",
    product_metadata.shape
)

display(
    product_metadata[
        [
            "faiss_row_id",
            "product_id",
            "product_name",
            "source"
        ]
    ].head()
)

SAVED: /kaggle/working/step11/product_embedding_metadata.parquet
SHAPE: (164926, 17)


,faiss_row_id,product_id,product_name,source
0,0,amazon_B07PXGQC1Q,Apple AirPods (2nd Generation) Wireless Earbud...,amazon
1,1,amazon_B07KTYJ769,"Amazon Smart Plug, for home automation, Works ...",amazon
2,2,amazon_B0BGNG1294,"Amazon Basics HDMI Cable, 18Gbps High-Speed, 4...",amazon
3,3,amazon_B08CLNX58K,SanDisk 32GB 2-Pack Ultra MicroSDHC UHS-I Memo...,amazon
4,4,amazon_B08WJSHSLC,Fire TV Stick (3rd Gen) with Alexa Voice Remot...,amazon


# Tạo embedding theo batch và lưu checkpoint

In [14]:
import gc
import time
import numpy as np
import torch

from pathlib import Path
from tqdm.auto import tqdm

CHECKPOINT_DIR = (
    OUT_DIR / "embedding_checkpoints"
)

CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

CHUNK_SIZE = 4096
BATCH_SIZE = 64

texts = (
    product_metadata["embedding_text_v2"]
    .tolist()
)

TOTAL = len(texts)

EXPECTED_DIM = 384

num_chunks = (
    TOTAL + CHUNK_SIZE - 1
) // CHUNK_SIZE

print("TOTAL PRODUCTS:", TOTAL)
print("CHUNK SIZE:", CHUNK_SIZE)
print("TOTAL CHUNKS:", num_chunks)
print("BATCH SIZE:", BATCH_SIZE)

start_time = time.time()

for chunk_id in tqdm(
    range(num_chunks),
    desc="Creating product embeddings"
):

    start = chunk_id * CHUNK_SIZE

    end = min(
        start + CHUNK_SIZE,
        TOTAL
    )

    checkpoint_path = (
        CHECKPOINT_DIR
        / f"product_chunk_{chunk_id:04d}.npy"
    )

    expected_shape = (
        end - start,
        EXPECTED_DIM
    )

    # Nếu checkpoint đã tồn tại và hợp lệ thì bỏ qua
    if checkpoint_path.exists():

        try:
            existing = np.load(
                checkpoint_path,
                mmap_mode="r"
            )

            if (
                existing.shape == expected_shape
                and existing.dtype == np.float32
            ):
                print(
                    f"SKIP chunk {chunk_id}: "
                    f"{start:,} → {end:,}"
                )
                continue

        except Exception:
            pass

        checkpoint_path.unlink()

    chunk_texts = texts[start:end]

    chunk_embeddings = model.encode(
        chunk_texts,
        batch_size=BATCH_SIZE,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False
    )

    chunk_embeddings = np.asarray(
        chunk_embeddings,
        dtype=np.float32
    )

    assert (
        chunk_embeddings.shape
        == expected_shape
    )

    assert np.isfinite(
        chunk_embeddings
    ).all()

    # Ghi file tạm trước để tránh checkpoint bị hỏng
    # nếu Kaggle ngắt phiên giữa lúc lưu.
    temp_path = checkpoint_path.with_suffix(
        ".tmp"
    )

    with open(temp_path, "wb") as f:
        np.save(
            f,
            chunk_embeddings
        )

    temp_path.replace(
        checkpoint_path
    )

    del chunk_embeddings

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print(
        f"SAVED chunk {chunk_id + 1}/{num_chunks}: "
        f"{start:,} → {end:,}"
    )

print("\nEMBEDDING FINISHED")

print(
    "TOTAL TIME:",
    round(
        (time.time() - start_time) / 60,
        2
    ),
    "minutes"
)

TOTAL PRODUCTS: 164926
CHUNK SIZE: 4096
TOTAL CHUNKS: 41
BATCH SIZE: 64


Creating product embeddings:   0%|          | 0/41 [00:00<?, ?it/s]

SAVED chunk 1/41: 0 → 4,096
SAVED chunk 2/41: 4,096 → 8,192
SAVED chunk 3/41: 8,192 → 12,288
SAVED chunk 4/41: 12,288 → 16,384
SAVED chunk 5/41: 16,384 → 20,480
SAVED chunk 6/41: 20,480 → 24,576
SAVED chunk 7/41: 24,576 → 28,672
SAVED chunk 8/41: 28,672 → 32,768
SAVED chunk 9/41: 32,768 → 36,864
SAVED chunk 10/41: 36,864 → 40,960
SAVED chunk 11/41: 40,960 → 45,056
SAVED chunk 12/41: 45,056 → 49,152
SAVED chunk 13/41: 49,152 → 53,248
SAVED chunk 14/41: 53,248 → 57,344
SAVED chunk 15/41: 57,344 → 61,440
SAVED chunk 16/41: 61,440 → 65,536
SAVED chunk 17/41: 65,536 → 69,632
SAVED chunk 18/41: 69,632 → 73,728
SAVED chunk 19/41: 73,728 → 77,824
SAVED chunk 20/41: 77,824 → 81,920
SAVED chunk 21/41: 81,920 → 86,016
SAVED chunk 22/41: 86,016 → 90,112
SAVED chunk 23/41: 90,112 → 94,208
SAVED chunk 24/41: 94,208 → 98,304
SAVED chunk 25/41: 98,304 → 102,400
SAVED chunk 26/41: 102,400 → 106,496
SAVED chunk 27/41: 106,496 → 110,592
SAVED chunk 28/41: 110,592 → 114,688
SAVED chunk 29/41: 114,688 → 11

# Kiểm tra toàn bộ checkpoint

In [15]:
checkpoint_files = sorted(
    CHECKPOINT_DIR.glob(
        "product_chunk_*.npy"
    )
)

print(
    "CHECKPOINT FILES:",
    len(checkpoint_files)
)

print(
    "EXPECTED CHUNKS:",
    num_chunks
)

assert len(checkpoint_files) == num_chunks, (
    "Chưa tạo đủ checkpoint."
)

total_vectors = 0

for chunk_id in range(num_chunks):

    path = (
        CHECKPOINT_DIR
        / f"product_chunk_{chunk_id:04d}.npy"
    )

    assert path.exists(), (
        f"Thiếu checkpoint {chunk_id}"
    )

    arr = np.load(
        path,
        mmap_mode="r"
    )

    expected_rows = min(
        CHUNK_SIZE,
        TOTAL - chunk_id * CHUNK_SIZE
    )

    assert arr.shape == (
        expected_rows,
        EXPECTED_DIM
    )

    assert arr.dtype == np.float32

    assert np.isfinite(arr).all()

    total_vectors += len(arr)

print("TOTAL VECTORS:", total_vectors)

assert total_vectors == len(
    product_metadata
)

print(
    "PASS: Tất cả embedding "
    "đã được tạo và lưu thành công."
)

CHECKPOINT FILES: 41
EXPECTED CHUNKS: 41
TOTAL VECTORS: 164926
PASS: Tất cả embedding đã được tạo và lưu thành công.


In [16]:
import numpy as np
from pathlib import Path

EMBEDDINGS_PATH = OUT_DIR / "product_embeddings.npy"

embeddings = np.lib.format.open_memmap(
    EMBEDDINGS_PATH,
    mode="w+",
    dtype=np.float32,
    shape=(TOTAL, EXPECTED_DIM)
)

for chunk_id in range(num_chunks):
    start = chunk_id * CHUNK_SIZE
    end = min(start + CHUNK_SIZE, TOTAL)

    checkpoint_path = (
        CHECKPOINT_DIR
        / f"product_chunk_{chunk_id:04d}.npy"
    )

    chunk = np.load(checkpoint_path)

    assert chunk.shape == (
        end - start,
        EXPECTED_DIM
    )

    embeddings[start:end] = chunk

embeddings.flush()
del embeddings

# Đọc lại để xác nhận file lưu thành công
embeddings = np.load(
    EMBEDDINGS_PATH,
    mmap_mode="r"
)

print("EMBEDDINGS:", embeddings.shape)
print("DTYPE:", embeddings.dtype)
print("FILE:", EMBEDDINGS_PATH)

assert embeddings.shape == (164_926, 384)
assert np.isfinite(embeddings).all()

print("PASS: Đã ghép toàn bộ embedding.")

EMBEDDINGS: (164926, 384)
DTYPE: float32
FILE: /kaggle/working/step11/product_embeddings.npy
PASS: Đã ghép toàn bộ embedding.


# Tạo FAISS index mới

In [17]:
import faiss
import numpy as np

FAISS_PATH = OUT_DIR / "product_faiss.index"

index = faiss.IndexFlatIP(EXPECTED_DIM)

for start in range(0, TOTAL, CHUNK_SIZE):
    end = min(start + CHUNK_SIZE, TOTAL)

    batch = np.ascontiguousarray(
        embeddings[start:end],
        dtype=np.float32
    )

    index.add(batch)

print("FAISS DIMENSION:", index.d)
print("FAISS VECTORS:", index.ntotal)

assert index.d == 384
assert index.ntotal == len(product_metadata)

faiss.write_index(
    index,
    str(FAISS_PATH)
)

print("SAVED:", FAISS_PATH)
print("PASS: FAISS index đã tạo thành công.")

FAISS DIMENSION: 384
FAISS VECTORS: 164926
SAVED: /kaggle/working/step11/product_faiss.index
PASS: FAISS index đã tạo thành công.


# Thử tìm kiếm Top 5 sản phẩm

In [18]:
def search_products(query, top_k=5):
    query_vector = model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype(np.float32)

    scores, row_ids = index.search(
        query_vector,
        top_k
    )

    results = product_metadata.iloc[
        row_ids[0]
    ].copy()

    results.insert(
        0,
        "similarity_score",
        scores[0]
    )

    return results[
        [
            "similarity_score",
            "product_id",
            "product_name",
            "source",
            "price",
            "currency",
            "average_rating"
        ]
    ].reset_index(drop=True)


query = (
    "Tôi cần thẻ nhớ microSD 32GB "
    "để lưu video và sử dụng cho điện thoại"
)

print("QUERY:", query)

display(
    search_products(
        query,
        top_k=5
    )
)

QUERY: Tôi cần thẻ nhớ microSD 32GB để lưu video và sử dụng cho điện thoại


,similarity_score,product_id,product_name,source,price,currency,average_rating
0,0.835000,amazon_B074RNRM2B,SanDisk 400GB Ultra MicroSDXC UHS-I Memory Car...,amazon,62.20,USD,4.8
1,0.819043,amazon_B011BRUOMO,SanDisk Ultra 32GB microSDHC UHS-I Card with A...,amazon,8.99,USD,4.6
2,0.815231,amazon_B0BRZPCQNJ,SanDisk 64GB 2-Pack Ultra microSDXC UHS-I Memo...,amazon,16.99,USD,4.7
3,0.814113,amazon_B0BRZG5TK4,SanDisk 256GB Ultra microSDXC UHS-I Memory Car...,amazon,25.49,USD,4.7
4,0.813782,amazon_B073JWXGNT,SanDisk 32GB Ultra MicroSDHC UHS-I Memory Card...,amazon,8.66,USD,4.7


# Lưu cấu hình và kiểm tra output

In [19]:
import json

config = {
    "step": "step11_product_embedding_faiss",
    "embedding_model": MODEL_NAME,
    "embedding_dimension": EXPECTED_DIM,
    "model_max_sequence_length": MAX_TOKENS,
    "embedding_text_column": "embedding_text_v2",
    "normalization": True,
    "faiss_index_type": "IndexFlatIP",
    "similarity": "cosine",
    "total_products": int(TOTAL),
    "chunk_size": CHUNK_SIZE,
    "batch_size": BATCH_SIZE,
    "source_dataset": "step10_unified_products",
    "faiss_mapping_column": "faiss_row_id",
}

CONFIG_PATH = (
    OUT_DIR / "step11_config.json"
)

with open(
    CONFIG_PATH,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        config,
        f,
        ensure_ascii=False,
        indent=2
    )

required_files = [
    "product_embeddings.npy",
    "product_faiss.index",
    "product_embedding_metadata.parquet",
    "step11_config.json",
]

print("===== STEP 11 OUTPUT =====")

for filename in required_files:
    path = OUT_DIR / filename

    assert path.exists(), (
        f"Thiếu file: {filename}"
    )

    print(
        filename,
        "|",
        round(
            path.stat().st_size / 1024**2,
            2
        ),
        "MB"
    )

print("\nPASS: Step 11 output đầy đủ.")

===== STEP 11 OUTPUT =====
product_embeddings.npy | 241.59 MB
product_faiss.index | 241.59 MB
product_embedding_metadata.parquet | 215.11 MB
step11_config.json | 0.0 MB

PASS: Step 11 output đầy đủ.


In [20]:
from pathlib import Path
import zipfile

OUT_DIR = Path("/kaggle/working/step11")
ZIP_PATH = Path("/kaggle/working/step11_final.zip")

required_files = [
    "product_embeddings.npy",
    "product_faiss.index",
    "product_embedding_metadata.parquet",
    "step11_config.json",
]

with zipfile.ZipFile(
    ZIP_PATH,
    mode="w",
    compression=zipfile.ZIP_STORED
) as zf:

    for filename in required_files:
        path = OUT_DIR / filename

        assert path.exists(), f"Thiếu file: {filename}"

        zf.write(
            path,
            arcname=filename
        )

print("SAVED:", ZIP_PATH)
print(
    "ZIP SIZE:",
    round(ZIP_PATH.stat().st_size / 1024**2, 2),
    "MB"
)

with zipfile.ZipFile(ZIP_PATH) as zf:
    print("\nFILES IN ZIP:")
    for name in zf.namelist():
        print("-", name)

print("\nPASS: STEP 11 PACKAGED")

SAVED: /kaggle/working/step11_final.zip
ZIP SIZE: 698.29 MB

FILES IN ZIP:
- product_embeddings.npy
- product_faiss.index
- product_embedding_metadata.parquet
- step11_config.json

PASS: STEP 11 PACKAGED
